# Projeto de Python para Finanças

In [1]:
# instalação
# pip install yfinance pandas numpy plotly nbformat

### Parte 1: Obter cotações e construção de carteira

In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime,timedelta

acoes = ["ITUB4.SA","PETR4.SA","VALE3.SA","IVVB11.SA"]
indices = ["^BVSP","^GSPC","BRL=X", "GC=F"]
#índices = ibovespa, sp500, dolar, ouro

data_inicio = datetime.now()-timedelta(days=365)
data_inicio = data_inicio.strftime(r'%Y-%m-%d')
data_fim = datetime.now().strftime(r'%Y-%m-%d')


def pegar_cotacoes(lista_tickers,data_inicio,data_fim):
    df =yf.download(lista_tickers,data_inicio,data_fim, auto_adjust=False)
    df=df['Adj Close']
    df=df.ffill()
    return df

lista_tickers = acoes+indices

df_cotacoes = pegar_cotacoes(lista_tickers,data_inicio,data_fim)

display(df_cotacoes)

[*********************100%***********************]  8 of 8 completed


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2025-04-15,5.8542,3218.699951,29.104784,355.750000,27.836357,49.399677,129245.0,5396.629883
2025-04-16,5.8846,3326.600098,29.060341,346.399994,27.836357,48.252129,128317.0,5275.700195
2025-04-17,5.8654,3308.699951,29.113674,343.750000,28.386726,48.545902,129650.0,5282.700195
2025-04-21,5.8654,3406.199951,29.113674,343.750000,28.386726,48.545902,129650.0,5158.200195
2025-04-22,5.8071,3400.800049,29.682617,339.250000,28.451139,49.408855,130464.0,5287.759766
...,...,...,...,...,...,...,...,...
2026-04-09,5.1002,4792.200195,45.750000,388.570007,47.900002,84.690002,195129.0,6824.660156
2026-04-10,5.0958,4761.899902,46.070000,384.700012,49.029999,85.589996,197324.0,6816.890137
2026-04-13,5.0050,4742.399902,45.830002,387.649994,49.779999,87.360001,198001.0,6886.240234


In [3]:
dic_carteira = {
    'ITUB4.SA':5000,
    'VALE3.SA':3000,
    'PETR4.SA':4000,
    'IVVB11.SA':6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())

for ativo in dic_carteira:
    preco_inicial = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo]/preco_inicial
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes

df_carteira["Total"] = df_carteira.sum(axis=1)
display(df_carteira)

,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2025-04-15,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2025-04-16,4992.364978,2930.310354,4000.000000,5842.304887,17764.980219
2025-04-17,5001.527266,2948.150976,4079.086392,5797.610682,17826.375316
2025-04-21,5001.527266,2948.150976,4079.086392,5797.610682,17826.375316
2025-04-22,5099.267731,3000.557382,4088.342355,5721.714687,17909.882155
...,...,...,...,...,...
2026-04-09,7859.532643,5143.151157,6883.084784,6553.534909,26439.303492
2026-04-10,7914.506371,5197.807014,7045.461958,6488.264436,26646.039778
2026-04-13,7873.276402,5305.297854,7153.234681,6538.018168,26869.827105


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [4]:
df_cotacoes["SP500(R$)"] = df_cotacoes["^GSPC"] * df_cotacoes["BRL=X"]
df_cotacoes["Ouro (R$)"] = df_cotacoes["GC=F"] * df_cotacoes["BRL=X"]
df_cotacoes["Dolar"] = df_cotacoes["BRL=X"]

df_cotacoes = df_cotacoes.drop(columns=["^GSPC","GC=F","BRL=X"])

display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500(R$),Ouro (R$),Dolar
Date,,,,,,,,
2025-04-15,29.104784,355.750000,27.836357,49.399677,129245.0,31592.950046,18842.912888,5.8542
2025-04-16,29.060341,346.399994,27.836357,48.252129,128317.0,31045.386227,19575.711475,5.8846
2025-04-17,29.113674,343.750000,28.386726,48.545902,129650.0,30985.148867,19406.848156,5.8654
2025-04-21,29.113674,343.750000,28.386726,48.545902,129650.0,30254.906587,19978.724640,5.8654
2025-04-22,29.682617,339.250000,28.451139,49.408855,130464.0,30706.548779,19748.785349,5.8071
...,...,...,...,...,...,...,...,...
2026-04-09,45.750000,388.570007,47.900002,84.690002,195129.0,34807.132932,24441.180281,5.1002
2026-04-10,46.070000,384.700012,49.029999,85.589996,197324.0,34737.508233,24265.689155,5.0958
2026-04-13,45.830002,387.649994,49.779999,87.360001,198001.0,34465.633161,23735.712054,5.0050


In [5]:
def calcular_retorno(df):
    retorno = df.iloc[-1]/df.iloc[0] -1
    return retorno

display(calcular_retorno(df_carteira))
display(calcular_retorno(df_cotacoes))


ITUB4.SA     0.598706
VALE3.SA     0.787461
PETR4.SA     0.720053
IVVB11.SA    0.099930
Total        0.490872
dtype: float64

Ticker
ITUB4.SA     0.598706
IVVB11.SA    0.099930
PETR4.SA     0.720053
VALE3.SA     0.787461
^BVSP        0.537058
SP500(R$)    0.101445
Ouro (R$)    0.272262
Dolar       -0.146869
dtype: float64

In [6]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao["Carteira"] = df_carteira["Total"]
display(df_comparacao)
display(calcular_retorno(df_comparacao))

Ticker,^BVSP,SP500(R$),Ouro (R$),Dolar,Carteira
Date,,,,,
2025-04-15,129245.0,31592.950046,18842.912888,5.8542,18000.000000
2025-04-16,128317.0,31045.386227,19575.711475,5.8846,17764.980219
2025-04-17,129650.0,30985.148867,19406.848156,5.8654,17826.375316
2025-04-21,129650.0,30254.906587,19978.724640,5.8654,17826.375316
2025-04-22,130464.0,30706.548779,19748.785349,5.8071,17909.882155
...,...,...,...,...,...
2026-04-09,195129.0,34807.132932,24441.180281,5.1002,26439.303492
2026-04-10,197324.0,34737.508233,24265.689155,5.0958,26646.039778
2026-04-13,198001.0,34465.633161,23735.712054,5.0050,26869.827105


Ticker
^BVSP        0.537058
SP500(R$)    0.101445
Ouro (R$)    0.272262
Dolar       -0.146869
Carteira     0.490872
dtype: float64

In [7]:
import plotly.express as px
df_comparacao = df_comparacao / df_comparacao.iloc[0] -1
#rentabilidade acumulada!
display(df_comparacao)

grafico = px.line(df_comparacao,x=df_comparacao.index,y=df_comparacao.columns)

grafico.show()

Ticker,^BVSP,SP500(R$),Ouro (R$),Dolar,Carteira
Date,,,,,
2025-04-15,0.000000,0.000000,0.000000,0.000000,0.000000
2025-04-16,-0.007180,-0.017332,0.038890,0.005193,-0.013057
2025-04-17,0.003134,-0.019239,0.029928,0.001913,-0.009646
2025-04-21,0.003134,-0.042353,0.060278,0.001913,-0.009646
2025-04-22,0.009432,-0.028057,0.048075,-0.008046,-0.005007
...,...,...,...,...,...
2026-04-09,0.509761,0.101737,0.297102,-0.128796,0.468850
2026-04-10,0.526744,0.099534,0.287789,-0.129548,0.480336
2026-04-13,0.531982,0.090928,0.259663,-0.145058,0.492768


### Parte 3: Análise de Risco

In [10]:
#aqui começam as magias de análise gráfica, só passe por isso
#Correlação
import numpy as np
tabela_rentabilidade_diaria = df_cotacoes /df_cotacoes.shift(1)
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()

tabela_correlacao = tabela_rentabilidade_diaria.corr()

grafico_correlacao = px.imshow(tabela_correlacao, text_auto=True, color_continuous_scale="greens")
grafico_correlacao.update_layout(template="plotly_dark")
grafico_correlacao.show()

#Variância de retorno
tabela_volatilidade = tabela_rentabilidade_diaria.std()*np.sqrt(252)

display(tabela_volatilidade)

Ticker
ITUB4.SA     0.222886
IVVB11.SA    0.124409
PETR4.SA     0.232024
VALE3.SA     0.227694
^BVSP        0.158163
SP500(R$)    0.171912
Ouro (R$)    0.296231
Dolar        0.113125
dtype: float64

### Parte 4: Análise Técnica e Indicadores

In [16]:
import plotly.graph_objects as go

ticker = "VALE3.SA"
df =yf.download(ticker,"2020-01-01","2025-12-31",multi_level_index=False)

#medias móveis
df["MM50"] = df["Close"].rolling(50).mean()
df["MM200"] = df["Close"].rolling(200).mean()

grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=df.index,open=df["Open"],close=df["Close"],high=df["High"],low=df["Low"], name="Price"))
grafico.add_trace(go.Scatter(x=df.index,y=df["MM50"], name="MM50", line={"color":"blue", "width":1}))
grafico.add_trace(go.Scatter(x=df.index,y=df["MM200"], name="MM200", line={"color":"purple", "width":1}))
grafico.show()


[*********************100%***********************]  1 of 1 completed
